<a href="https://colab.research.google.com/github/ghirr/Smart-Traffic-Flow-Prediction-and-Optimization/blob/main/Part3_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Partie 3 : Réseaux de neurones artificiels (ANN)

**Projet : Smart Traffic Flow Prediction and Optimization**

Même problème, mêmes jeux et mêmes métriques que la partie 2 : prédire, pour chaque cellule de 100 m et pour la **minute suivante**, l'état du trafic (`etat`), la vitesse moyenne (`vitesse`) et le flux (`flux`). Cela permet une comparaison équitable avec le Bagging et le Boosting (partie 5).

**Plan** : 1. rappels et choix d'architecture · 2. sous-apprentissage, surapprentissage et bon réglage · 3. optimisation des hyperparamètres · 4. évaluation finale · 5. analyse.

## 1. Rappels et choix de conception

**Un réseau de neurones « perceptron multicouche » (MLP)** est une succession de couches. Chaque neurone calcule une somme pondérée de ses entrées, puis applique une **fonction d'activation** non linéaire. L'apprentissage ajuste les poids pour minimiser une **fonction de perte**, par **descente de gradient** (rétropropagation), en parcourant les données par petits groupes (*batchs*) pendant plusieurs passages (*epochs*).

| Élément | Choix retenu | Justification |
|---|---|---|
| Type de réseau | **MLP** (couches denses) | les données sont **tabulaires** : 39 variables par ligne. L'historique (retards, moyennes glissantes) et le voisinage spatial sont déjà résumés par le feature engineering, donc pas besoin de réseau récurrent ni convolutif |
| Entrées | variables **normalisées** (`*_scaled.parquet`) | un réseau est sensible à l'échelle des variables, contrairement aux arbres |
| Couches cachées | 2 à 3 couches de 32 à 256 neurones (à optimiser) | assez de capacité pour des relations non linéaires, sans excès pour 37 000 exemples |
| Activation cachée | **ReLU** (comparée à ELU et tanh) | simple, évite la disparition du gradient |
| Sortie, classification | 3 neurones, **softmax** | probabilité de chaque état |
| Sortie, régression | 1 neurone, **linéaire** | valeur réelle |
| Perte | classification : **entropie croisée** ; régression : **erreur quadratique moyenne (MSE)** | cohérent avec les métriques (F1, RMSE) |
| Optimiseur | **Adam** + réduction du taux d'apprentissage sur plateau | convergence rapide et peu sensible au réglage |
| Régularisation | **Dropout**, pénalité **L2**, **arrêt anticipé** | limiter le surapprentissage |
| Cibles de régression | **standardisées** (moyenne 0, écart-type 1, calculées sur le train) | entraînement plus stable ; on remet à l'échelle pour les métriques |

In [ ]:
import os, sys, json, time, warnings, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
SEED = 42
np.random.seed(SEED)

# ---- Données préparées par la partie 1 (dossier data_prepared/ du dépôt) ----
REPO_URL = "https://github.com/ghirr/Smart-Traffic-Flow-Prediction-and-Optimization.git"
BRANCH   = "main"                      # branche du dépôt qui contient data_prepared/ et results/
if not os.path.exists("data_prepared/meta.json") and "google.colab" in sys.modules:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
assert os.path.exists("data_prepared/meta.json"), "Dossier data_prepared/ introuvable : exécutez la partie 1 ou clonez le dépôt."
os.makedirs("results", exist_ok=True)

META   = json.load(open("data_prepared/meta.json"))
FEATS  = META["features"]
ETATS  = META["etats"]                 # ['congestionné', 'ralenti', 'fluide']
SEUILS = META["seuils_etat_m_s"]
SPLITS = ("train", "val", "test")

def charger(suffixe):
    """Retourne les trois jeux (train, val, test) : suffixe = 'raw' (arbres) ou 'scaled' (ANN)."""
    return [pd.read_parquet(f"data_prepared/{s}_{suffixe}.parquet") for s in SPLITS]

# Les trois tâches de prédiction (cibles de la minute suivante)
TASKS = {
    "etat":    dict(target="etat_next_code", kind="clf", nom="État du trafic"),
    "vitesse": dict(target="speed_next",     kind="reg", nom="Vitesse moyenne (m/s)"),
    "flux":    dict(target="n_veh_next",     kind="reg", nom="Flux (nb de véhicules)"),
}

from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
                             f1_score, recall_score, roc_auc_score, confusion_matrix)

def metriques(kind, y, pred, proba=None):
    """Métriques adaptées à la tâche : régression (MAE, RMSE, R²) ou classification (accuracy, F1 macro, rappels, AUC)."""
    if kind == "reg":
        return {"MAE": mean_absolute_error(y, pred), "RMSE": mean_squared_error(y, pred) ** 0.5, "R2": r2_score(y, pred)}
    rec = recall_score(y, pred, average=None, labels=[0, 1, 2])
    out = {"Accuracy": accuracy_score(y, pred), "F1_macro": f1_score(y, pred, average="macro"),
           "Rappel_congestionné": rec[0], "Rappel_ralenti": rec[1], "Rappel_fluide": rec[2]}
    if proba is not None:
        out["AUC_ovr"] = roc_auc_score(y, proba, multi_class="ovr")
    return out

def perte(kind, y, pred):
    """Critère unique de sélection (à minimiser) : MAE en régression, 1 - F1 macro en classification."""
    return mean_absolute_error(y, pred) if kind == "reg" else 1 - f1_score(y, pred, average="macro")

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import ParameterSampler
print("TensorFlow", tf.__version__)

SCALED = charger("scaled")

def X_y(df, tache):
    d = df.dropna(subset=[tache["target"]])
    y = d[tache["target"]].astype(int) if tache["kind"] == "clf" else d[tache["target"]].astype(float)
    return d[FEATS].values.astype("float32"), y.values

DATA = {t: {s: X_y(df, TASKS[t]) for s, df in zip(SPLITS, SCALED)} for t in TASKS}
STAT = {t: (DATA[t]["train"][1].mean(), DATA[t]["train"][1].std()) for t in TASKS if TASKS[t]["kind"] == "reg"}
print({t: {s: len(DATA[t][s][1]) for s in SPLITS} for t in TASKS})

In [ ]:
def construire(n_in, kind, couches=2, unites=64, activation="relu", dropout=0.0, l2=0.0, lr=1e-3):
    """MLP : [Dense -> (Dropout)] x couches, puis couche de sortie."""
    inp = keras.Input((n_in,))
    x = inp
    for _ in range(couches):
        x = layers.Dense(unites, activation=activation, kernel_regularizer=regularizers.l2(l2) if l2 else None)(x)
        if dropout: x = layers.Dropout(dropout)(x)
    if kind == "clf":
        out = layers.Dense(3, activation="softmax")(x)
        model = keras.Model(inp, out)
        model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    else:
        out = layers.Dense(1)(x)
        model = keras.Model(inp, out)
        model.compile(optimizer=keras.optimizers.Adam(lr), loss="mse", metrics=["mae"])
    return model

def cible(tache, y):
    """Cible vue par le réseau : standardisée en régression."""
    if TASKS[tache]["kind"] == "reg":
        mu, sd = STAT[tache]; return ((y - mu) / sd).astype("float32")
    return y

def entrainer(tache, seed=SEED, epochs=80, batch=256, patience=8, arret=True, **arch):
    """Entraîne un MLP. arret=True : arrêt anticipé + réduction du taux d'apprentissage ; False : toutes les epochs."""
    tf.keras.utils.set_random_seed(seed)
    kind = TASKS[tache]["kind"]
    (Xa, ya), (Xb, yb) = DATA[tache]["train"], DATA[tache]["val"]
    model = construire(Xa.shape[1], kind, **arch)
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=max(2, patience // 2))] if arret else []
    t0 = time.time()
    h = model.fit(Xa, cible(tache, ya), validation_data=(Xb, cible(tache, yb)), epochs=epochs, batch_size=batch,
                  callbacks=cb, verbose=0)
    return model, pd.DataFrame(h.history), time.time() - t0

def predire(model, tache, X):
    p = model.predict(X, verbose=0, batch_size=2048)
    if TASKS[tache]["kind"] == "clf": return p.argmax(1), p
    mu, sd = STAT[tache]; return p.ravel() * sd + mu, None

def evaluer(model, tache, split):
    X, y = DATA[tache][split]
    pred, proba = predire(model, tache, X)
    return metriques(TASKS[tache]["kind"], y, pred, proba)

## 2. Sous-apprentissage, surapprentissage et bon réglage

On entraîne trois réseaux sur la même tâche (`vitesse`) pour voir les trois régimes de la courbe d'apprentissage :

| Réseau | Description | Comportement attendu |
|---|---|---|
| **A, trop simple** | 1 couche de 2 neurones | **sous-apprentissage** : erreur élevée sur train *et* validation |
| **B, trop puissant** | 3 couches de 512 neurones, sans régularisation, sans arrêt, 150 epochs | **surapprentissage** : l'erreur de train descend, celle de validation remonte |
| **C, bien réglé** | 2 couches de 64, dropout, L2, arrêt anticipé | les deux courbes se rejoignent et se stabilisent |

In [ ]:
T = "vitesse"
reseaux = {
    "A : trop simple (1x2)":          dict(arch=dict(couches=1, unites=2),  epochs=60,  arret=False),
    "B : trop puissant (3x512)":      dict(arch=dict(couches=3, unites=512, lr=1e-3), epochs=150, arret=False),
    "C : régularisé (2x64, dropout, L2)": dict(arch=dict(couches=2, unites=64, dropout=0.2, l2=1e-4), epochs=100, arret=True),
}
courbes = {}
for nom, cfg in reseaux.items():
    model, hist, tps = entrainer(T, epochs=cfg["epochs"], arret=cfg["arret"], **cfg["arch"])
    courbes[nom] = hist
    mu, sd = STAT[T]
    print(f"{nom:38s} MAE train = {evaluer(model, T, 'train')['MAE']:.3f} | MAE val = {evaluer(model, T, 'val')['MAE']:.3f}  ({len(hist)} epochs, {tps:.0f} s)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8), sharey=True)
for a, (nom, h) in zip(ax, courbes.items()):
    a.plot(h["loss"], label="train"); a.plot(h["val_loss"], label="validation")
    a.set_title(nom, fontsize=10); a.set_xlabel("epoch"); a.legend()
ax[0].set_ylabel("perte MSE (cible standardisée)"); ax[0].set_ylim(0, None)
plt.tight_layout(); plt.show()

**Lecture.** Le réseau A n'a pas assez de capacité : les deux erreurs restent hautes (**sous-apprentissage**, biais élevé). Le réseau B mémorise le train : sa perte d'entraînement continue de baisser pendant que la perte de validation **remonte** (**surapprentissage**, variance élevée). Le réseau C, avec dropout, pénalité L2 et arrêt anticipé, garde les deux courbes proches. C'est ce type de réglage que l'on optimise ensuite.

## 3. Optimisation des hyperparamètres

Recherche aléatoire sur la validation, pour chacune des trois tâches. Hyperparamètres : nombre de couches, neurones par couche, activation, dropout, pénalité L2, taux d'apprentissage, taille de batch. Le critère est le même que pour les arbres (MAE en régression, 1 − F1 macro en classification).

In [ ]:
ESPACE = {"couches": [2, 3], "unites": [32, 64, 128, 256], "activation": ["relu", "elu", "tanh"],
          "dropout": [0.0, 0.1, 0.2, 0.3], "l2": [0.0, 1e-5, 1e-4, 1e-3], "lr": [3e-4, 1e-3, 3e-3]}
BATCHS = [128, 256, 512]
N_ESSAIS = 6

RECH, MODELES = {}, {}
for t in TASKS:
    kind = TASKS[t]["kind"]; Xb, yb = DATA[t]["val"]
    lignes, meilleur = [], (np.inf, None, None)
    for i, p in enumerate(ParameterSampler(ESPACE, N_ESSAIS, random_state=SEED)):
        batch = BATCHS[i % len(BATCHS)]
        model, hist, tps = entrainer(t, epochs=60, batch=batch, patience=8, **p)
        pred, _ = predire(model, t, Xb)
        sc = perte(kind, yb, pred)
        lignes.append({**p, "batch": batch, "epochs": len(hist), "perte_val": sc, "temps_s": tps})
        if sc < meilleur[0]: meilleur = (sc, {**p, "batch": batch}, i)
        keras.backend.clear_session()
    RECH[t] = pd.DataFrame(lignes).sort_values("perte_val").reset_index(drop=True)
    MODELES[t] = meilleur[1]
    print(f"\n=== {t} : essais classés (perte = {'1 - F1 macro' if kind == 'clf' else 'MAE'}) ===")
    print(RECH[t].round(4).to_string(index=False))

### Sensibilité aux hyperparamètres

Sur ces essais, quelles tendances voit-on ? (Le nombre d'essais est volontairement petit, pour un temps de calcul raisonnable : les tendances sont indicatives.)

In [ ]:
tous = pd.concat([RECH[t].assign(tache=t) for t in TASKS])
# Perte relative au meilleur essai de chaque tâche, pour pouvoir comparer des tâches différentes
tous["perte_relative"] = tous["perte_val"] / tous.groupby("tache")["perte_val"].transform("min")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for a, col in zip(ax, ["activation", "dropout", "lr"]):
    sns.boxplot(data=tous, x=col, y="perte_relative", ax=a, color="lightsteelblue")
    sns.stripplot(data=tous, x=col, y="perte_relative", ax=a, color="black", size=3)
    a.set_title(f"Effet de {col}"); a.set_ylabel("perte / meilleure perte de la tâche")
plt.tight_layout(); plt.show()
print(tous.groupby("activation")["perte_relative"].mean().round(3).to_string())

## 4. Évaluation finale

Les meilleurs réglages sont ré-entraînés avec **3 graines aléatoires** (un réseau dépend de l'initialisation des poids : on mesure donc la variabilité). Les métriques de la graine 42 sont utilisées pour la comparaison, avec les prédictions sauvegardées pour la partie 5. Le test n'est utilisé qu'ici.

In [ ]:
SEEDS = [42, 43, 44]
FINAL, TEMPS, preds, courbes_finales = {}, {}, {}, {}
for t in TASKS:
    kind = TASKS[t]["kind"]; p = dict(MODELES[t]); batch = p.pop("batch")
    essais = []
    for sd in SEEDS:
        model, hist, tps = entrainer(t, seed=sd, epochs=100, batch=batch, patience=10, **p)
        essais.append({"seed": sd, "val": evaluer(model, t, "val"), "test": evaluer(model, t, "test"), "epochs": len(hist)})
        if sd == SEEDS[0]:
            Xc, yc = DATA[t]["test"]
            preds[t] = {"y_true": yc, "ANN": predire(model, t, Xc)[0]}
            courbes_finales[t] = hist; TEMPS[t] = tps
        keras.backend.clear_session()
    FINAL[t] = {"params": MODELES[t], "essais": essais}

crit = {"etat": "F1_macro", "vitesse": "MAE", "flux": "MAE"}
rows = []
for t in TASKS:
    v = [e["val"][crit[t]] for e in FINAL[t]["essais"]]; te = [e["test"][crit[t]] for e in FINAL[t]["essais"]]
    rows.append({"tâche": t, "critère": crit[t], "val (graine 42)": v[0], "test (graine 42)": te[0],
                 "test moyenne 3 graines": np.mean(te), "test écart-type": np.std(te)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for a, t in zip(ax, TASKS):
    h = courbes_finales[t]
    a.plot(h["loss"], label="train"); a.plot(h["val_loss"], label="validation")
    a.set_title(f"Réseau final : {t} ({len(h)} epochs)"); a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Sauvegardes pour la partie 5 (même structure que la partie 2)
resultats = {t: {"ANN": {"test": FINAL[t]["essais"][0]["test"], "val": FINAL[t]["essais"][0]["val"], "temps_s": TEMPS[t],
                         "params": FINAL[t]["params"],
                         "test_3_graines": {crit[t]: [e["test"][crit[t]] for e in FINAL[t]["essais"]]}}} for t in TASKS}
json.dump(resultats, open("results/ann_metrics.json", "w"), indent=2, default=lambda o: float(o) if hasattr(o, "__float__") else str(o))
for t in TASKS:
    pd.DataFrame(preds[t]).to_parquet(f"results/ann_preds_{t}.parquet", index=False)
print("Sauvegardé : results/ann_metrics.json et results/ann_preds_*.parquet")

## 5. Analyse

### Ce que montrent les courbes d'apprentissage
- **Réseau A (1×2)** : les deux courbes se superposent et plafonnent haut (MAE vitesse 1,317 en train, 1,323 en validation). Le modèle est trop pauvre pour apprendre la structure : c'est du **sous-apprentissage**.
- **Réseau B (3×512, sans régularisation)** : l'erreur de train s'effondre (MAE 0,355) alors que celle de validation remonte dès les premières epochs (MAE 1,510). Le réseau mémorise les fenêtres d'entraînement : c'est du **surapprentissage**, et c'est aussi le plus lent à entraîner (212 s contre 22 s pour C).
- **Réseau C (2×64, dropout, L2, arrêt anticipé)** : train et validation se rejoignent (1,290 / 1,329) ; l'arrêt anticipé coupe l'entraînement à 49 epochs. C'est le bon régime : la validation n'est pas spectaculairement meilleure que celle de A (1,329 contre 1,323 en MAE), car les variables explicatives limitent déjà la performance, mais le modèle ne dérive pas.

### Hyperparamètres (6 essais par tâche, tendances indicatives)
- Les meilleurs réglages sont les mêmes pour les trois tâches : **3 couches, 128 neurones, tanh, dropout 0,3, pas d'apprentissage 3·10⁻³**. Sur la validation, les écarts entre bons essais sont petits (par exemple perte 0,266 à 0,274 pour l'état), signe que la performance est surtout limitée par l'information contenue dans les variables, pas par le réglage.
- Le **tanh** fait légèrement mieux que le **ReLU** (perte relative moyenne 1,020 contre 1,041) ; le seul essai en ReLU est aussi le moins bon pour la vitesse et le flux. Avec seulement 6 essais, on ne peut pas conclure de façon ferme.
- Les deux essais avec un pas d'apprentissage de 3·10⁻³ figurent parmi les 4 premiers pour chacune des trois tâches, et le meilleur de chaque tâche en fait partie : avec un entraînement limité à 60 epochs, un pas plus élevé converge mieux. Le nombre d'essais reste trop petit pour une conclusion ferme.

### Performance finale (test, moyenne de 3 graines)
| Tâche | Critère | Graine 42 | Moyenne 3 graines | Écart-type |
|---|---|---|---|---|
| État | F1 macro | 0,731 | 0,727 | 0,005 |
| Vitesse | MAE | 1,312 | 1,302 | 0,009 |
| Flux | MAE | 1,460 | 1,460 | 0,002 |

- L'**écart-type entre graines est faible** (inférieur à 0,01) : le résultat ne dépend pas d'un coup de chance sur l'initialisation.
- Les courbes finales : pour la vitesse et le flux, la perte de validation reste sous celle d'entraînement (effet du dropout, actif seulement à l'entraînement), sans remontée. Pour l'état, un petit écart apparaît (validation autour de 0,62 contre 0,60 en train) : léger début de surapprentissage, contenu par l'arrêt anticipé.
- **Comparaison avec les ensembles** (partie 2) : le réseau est au niveau de Random Forest et XGBoost sur l'état (F1 0,731 contre 0,730 pour les deux), un peu moins bon sur la vitesse (MAE 1,312 contre 1,285 pour RF et 1,280 pour XGBoost) et sur le flux (1,460 contre 1,474 pour RF et 1,445 pour XGBoost). La différence est de l'ordre de 2 % : **sur ces données tabulaires de taille moyenne (environ 37 000 lignes), le réseau n'apporte pas de gain par rapport au boosting**, mais coûte plus cher à régler et plus lent à entraîner (31 à 78 s contre 4 à 7 s).
- La classe **« ralenti »** (classe intermédiaire) reste la plus difficile (rappel 0,62), comme pour les ensembles : ses frontières avec « fluide » et « congestionné » sont floues, puisque les classes viennent de seuils (quantiles) sur une vitesse continue.